In [1]:
from src.core.market_calendar import USMarketsCalendar
from src.core.market_clock import MarketClock, AcceleratedTimeHeartbeat
from src.core.system_manager import SystemManager

from src.data.data_access import InMemorySessionDAL
from src.data.data_ingestion import InMemorySessionDIL

from src.monitoring.health_checks import build_readiness_checks


checks = build_readiness_checks(db_connector=None, ibkr_gateway=None, eodhd_client=None)


# heartbeat = SystemTimeHeartbeat()
sim_heartbeat = AcceleratedTimeHeartbeat()

market_clock = MarketClock(time_source=sim_heartbeat)
market_calendar = USMarketsCalendar()

data_ingestion = InMemorySessionDIL()
data_access = InMemorySessionDAL()


system_manager = SystemManager(market_clock=market_clock,
                               market_calendar=market_calendar,
                               data_ingestion=data_ingestion,
                               data_access=data_access)


# system_manager._market_clock.start()

system_manager._prepare_bootstrap(checks=checks)

system_manager.launch_global_runtime()

system_manager.launch_local_runtime()


In [2]:
sm = system_manager._modules.session_manager

In [3]:
key = list(sm._sessions.keys())[0]
key3 = list(sm._sessions.keys())[1]


In [4]:
sm._sessions[key].stack.portfolio.refresh_portfolio_state()
sm._sessions[key].stack.risk.refresh_risk_state()

In [5]:
sm._sessions[key].stack.portfolio.portfolio_state

PortfolioSnapshot(account_id='account_0', as_of=datetime.datetime(2026, 1, 18, 14, 31, 33, 157586, tzinfo=datetime.timezone.utc), base_currency='USD', cash=Decimal('100000.00'), positions={'AAPL': Position(symbol='AAPL', quantity=Decimal('10'), avg_price=Decimal('175.50')), 'MSFT': Position(symbol='MSFT', quantity=Decimal('5'), avg_price=Decimal('410.25'))}, snapshot_version=1)

In [6]:
sm._sessions[key].stack.risk.risk_state

RiskSnapshot(account_id='account_0', as_of=datetime.datetime(2026, 1, 18, 14, 31, 33, 157611, tzinfo=datetime.timezone.utc), stop_loss_pct_by_symbol={'AAPL': Decimal('0.05')}, stop_loss_price_by_symbol={'AAPL': Decimal('175.50')}, snapshot_version=1)